# Last Known Seconds: gerador de vídeo (Colab grátis)

1. Menu **Ambiente de execução → Executar tudo** (a GPU T4 já vem pedida neste notebook).
2. Quando pedir, **permita o acesso ao Google Drive** (obrigatório: cada clipe pronto fica salvo lá; se a sessão cair, é só rodar de novo que continua de onde parou).
3. **Deixe esta aba aberta** até o fim. O lote atual (`atual.txt`) é baixado sozinho do GitHub.
4. No final, o resultado fica em **Meu Drive → LastKnownSeconds** e o navegador também baixa `resultado_<lote>.zip`.


In [ ]:
#@title 1. Conferir a GPU
import subprocess
out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(out or 'SEM GPU')
assert 'GPU' in out, 'Sem GPU: Ambiente de execução > Alterar o tipo de ambiente de execução > T4 GPU > Salvar, e rode de novo.'

In [ ]:
#@title 2. Instalar as bibliotecas (cerca de 1 min)
!pip install -q "diffusers>=0.35.1" gguf spandrel imageio-ffmpeg ftfy sentencepiece accelerate
print('ok')

In [ ]:
#@title 3. Google Drive (OBRIGATÓRIO: cada clipe pronto fica salvo lá)
# Sem o Drive, tudo se perde se a aba fechar (aconteceu em 12/11). Clique em "Conectar ao Google Drive" e permita.
import os
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/LastKnownSeconds'
os.makedirs(ROOT, exist_ok=True)
print('progresso salvo em', ROOT)

In [ ]:
#@title 4. Baixar o lote atual (do GitHub)
import zipfile, shutil, glob, urllib.request
RAW = 'https://raw.githubusercontent.com/teeokuz/lks-colab/main'
LOTE = urllib.request.urlopen(RAW + '/atual.txt').read().decode().strip()
NOME = f'/content/{LOTE}.zip'
urllib.request.urlretrieve(f'{RAW}/lotes/{LOTE}.zip', NOME)
IN = '/content/lks/' + LOTE
shutil.rmtree(IN, ignore_errors=True)
zipfile.ZipFile(NOME).extractall(IN)
OUT = f'{ROOT}/{LOTE}'
os.makedirs(OUT, exist_ok=True)
print('lote', LOTE, '->', OUT)

In [ ]:
#@title 5. Gerar (de 30 min a 3 h; deixe a aba aberta)
!python "{IN}/lks_colab.py" "{IN}" "{OUT}"

In [ ]:
#@title 6. Baixar o resultado
import json
from google.colab import files
shots = json.load(open(f'{IN}/shots.json'))
novas = {s['id'] for s in shots if s.get('cands')}
z = f'/content/resultado_{LOTE}.zip'
with zipfile.ZipFile(z, 'w') as f:
    for p in glob.glob(f'{OUT}/final/*.mp4'):
        f.write(p, 'final/' + os.path.basename(p))
    for p in glob.glob(f'{OUT}/keyframes/*.png'):
        if os.path.basename(p).rsplit('_', 1)[0] in novas:
            f.write(p, 'keyframes/' + os.path.basename(p))
print(len(zipfile.ZipFile(z).namelist()), 'arquivos,', round(os.path.getsize(z) / 1e6), 'MB')
shutil.copy(z, ROOT)  # uma cópia fica no Drive
try:
    files.download(z)
except Exception:
    print('Baixe pelo Drive: Meu Drive/LastKnownSeconds/' + os.path.basename(z))